In [ ]:
def main(sh_path: str, sweep_arrays: list[str] = None, out_path: str = None):
    # 1. Set default iteration parameters if none are provided
    if sweep_arrays is None:
        sweep_arrays = ["models", "neighborhoods", "encodings", "num_layers", "hidden_channels"]
        
    print(f"Parsing parameters from {sh_path}...")
    
    with open(sh_path, 'r') as f:
        content = f.read()
        
    # 2. Extract the Hydra override keys mapping from SWEEP_CONFIG
    mapping = get_override_mapping(content)
    
    # 3. Dynamically extract values and keys for the target sweep arrays
    sweep_data = {}
    for arr in sweep_arrays:
        vals_raw, override_key = parse_bash_array(content, arr, mapping)
        sweep_data[arr] = {
            "override_key": override_key,
            "values": [get_hydra_val(v) for v in vals_raw]
        }
        
    # 4. Automatically fix any non-swept parameters to their first value
    fixed_params = {}
    fixed_overrides = []
    
    print("\n--- Fixed Parameters (Using 1st value from script) ---")
    for arr_name, override_key in mapping.items():
        if arr_name not in sweep_arrays:
            try:
                vals_raw, _ = parse_bash_array(content, arr_name, mapping)
                first_val = get_hydra_val(vals_raw[0])
                fixed_params[arr_name] = first_val
                fixed_overrides.append(f"{override_key}={first_val}")
                print(f"Fixed '{arr_name}': {first_val}")
            except Exception as e:
                # Catch variables defined in mapping but missing/commented out in the bash script
                print(f"Warning: Could not set fixed value for '{arr_name}'")
    print("-" * 54 + "\n")
    
    # 5. Generate Combinations dynamically based on sweep_arrays
    keys = list(sweep_data.keys())
    value_lists = [sweep_data[k]["values"] for k in keys]
    combinations = list(itertools.product(*value_lists))
    
    results = []
    print(f"Total combinations to evaluate: {len(combinations)}")
    print("-" * 60)

    # 6. Initialize Hydra API
    GlobalHydra.instance().clear()
    hydra.initialize(version_base="1.3", config_path="../configs") 

    for idx, combo in enumerate(combinations):
        # Zip the current combination back to their bash array names
        combo_dict = dict(zip(keys, combo))
        
        # Validation checks: Get datasets from current combo OR fallback to the fixed value
        model_val = combo_dict.get("models", "")
        dataset_val = combo_dict.get("datasets", fixed_params.get("datasets", ""))
        
        if model_val.startswith('cell/') and dataset_val.startswith('simplicial/'):
            print(f"[{idx+1}/{len(combinations)}] Skipping invalid combo (Cell model + Simplicial dataset)")
            continue

        # 7. Dynamically Build Hydra Overrides
        # A. Add the dynamic sweep values
        # 7. Dynamically Build Hydra Overrides
        overrides = []
        
        # A. Add the dynamic sweep values
        for k, v in combo_dict.items():
            override_key = sweep_data[k]['override_key']
            
            # Catch the special @@@ syntax used in transforms
            if "@@@" in str(v):
                # The first part is the value for the primary key
                parts = str(v).split("@@@")
                primary_val = parts[0].strip()
                overrides.append(f"{override_key}={primary_val}")
                
                # The remaining parts are separate key=value overrides
                for extra_override in parts[1:]:
                    if extra_override.strip():
                        overrides.append(extra_override.strip())
            else:
                overrides.append(f"{override_key}={v}")
        
        # B. Add the fixed values
        for arr_name, override_key in mapping.items():
            if arr_name not in sweep_arrays:
                fixed_val = fixed_params.get(arr_name, "")
                if fixed_val:
                    # Apply the same @@@ split logic to fixed parameters just in case
                    if "@@@" in str(fixed_val):
                        parts = str(fixed_val).split("@@@")
                        overrides.append(f"{override_key}={parts[0].strip()}")
                        for extra_override in parts[1:]:
                            if extra_override.strip():
                                overrides.append(extra_override.strip())
                    else:
                        overrides.append(f"{override_key}={fixed_val}")
            
        # C. Add environment/testing constants
        overrides.extend([
            "train=False", 
            "test=False",
            "trainer.accelerator=cpu",
            "trainer.devices=auto"
        ])

        try:
            cfg = hydra.compose(config_name="run.yaml", overrides=overrides)
            
            # --- DATALOADER INSTANTIATION ---
            dataset_loader = hydra.utils.instantiate(cfg.dataset.loader)
            loaded_dataset, dataset_dir = dataset_loader.load()
            
            transform_config = hydra.utils.instantiate(cfg.transforms) if cfg.get("transforms", None) else None
            
            preprocessor = PreProcessor(loaded_dataset, dataset_dir, transform_config)
            dataset_train, dataset_val, dataset_test = preprocessor.load_dataset_splits(cfg.dataset.split_params)
            
            datamodule = TBDataloader(
                dataset_train=dataset_train,
                dataset_val=dataset_val,
                dataset_test=dataset_test,
                **cfg.dataset.get("dataloader_params", {}),
            )
            
            instantiated_model = hydra.utils.instantiate(
                cfg.model,
                evaluator=cfg.evaluator,
                optimizer=cfg.optimizer,
                loss=cfg.loss,
            )
            
            total_params, breakdown = count_detailed_parameters(instantiated_model)
            
            # 8. Dynamically construct the result row
            res = {k: combo_dict[k] for k in keys} # Add the sweep values first
            res.update({
                "Total_Params": total_params,
                "Backbone_Params": breakdown["Backbone"],
                "Encoder_Params": breakdown["Feature_Encoder"],
                "Readout_Params": breakdown["Readout"],
                "Other_Params": breakdown["Other"]
            })
            results.append(res)
            
            # Create a quick log string based on the active sweep elements
            log_info = " ".join([f"{k[:3]}:{str(v).split('/')[-1]}" for k, v in combo_dict.items()])
            print(f"[{idx+1}/{len(combinations)}] SUCCESS | Total: {total_params:,} ({log_info})")

        except Exception as e:
            print(f"[{idx+1}/{len(combinations)}] FAILED  | Error: {e}")

    # 9. Export Results
    print("-" * 60)
    df = pd.DataFrame(results)
    
    if not df.empty and out_path is not None:
        # Sort dynamically by the first sweep parameter and then Backbone_Params
        df = df.sort_values(by=[keys[0], "Backbone_Params"])
        df.to_csv(out_path, index=False)
        print(f"\nSaved results to {out_path}")
    else:
        print("No successful combinations to report.")

In [ ]:
sh_path = "../scripts/hopse_m.sh"
out_path = "parameter_sweep_results_hopse_m.csv"
# You can now easily pass arbitrary combinations from the bash script!
target_arrays = ["models", "encodings", "num_layers", "hidden_channels"]
main(sh_path, target_arrays, out_path)

In [ ]:
sh_path = "../scripts/gcn.sh"
out_path = "parameter_sweep_results_gcn.csv"
# You can now easily pass arbitrary combinations from the bash script!
target_arrays = ["models", "transform_presets", "num_layers", "hidden_channels"]
main(sh_path, target_arrays, out_path)

Parsing parameters from ../scripts/gcn.sh...

--- Fixed Parameters (Using 1st value from script) ---
Fixed 'datasets': graph/MUTAG
Fixed 'proj_dropouts': 0.25
Fixed 'lrs': 0.01
Fixed 'weight_decays': 0
Fixed 'batch_sizes': 128
Fixed 'DATA_SEEDS': 0
------------------------------------------------------

Total combinations to evaluate: 18
------------------------------------------------------------
[1/18] SUCCESS | Total: 18,071 (mod:gcn tra:no_transform num:1 hid:128)
[2/18] SUCCESS | Total: 68,887 (mod:gcn tra:no_transform num:1 hid:256)
[3/18] SUCCESS | Total: 34,583 (mod:gcn tra:no_transform num:2 hid:128)
[4/18] SUCCESS | Total: 134,679 (mod:gcn tra:no_transform num:2 hid:256)
[5/18] SUCCESS | Total: 67,607 (mod:gcn tra:no_transform num:4 hid:128)
[6/18] SUCCESS | Total: 266,263 (mod:gcn tra:no_transform num:4 hid:256)


Processing...



Applying transforms to 188 graphs...


Processing graphs: 100%|██████████| 188/188 [00:02<00:00, 86.60graph/s]
Done!


[7/18] SUCCESS | Total: 24,228 (mod:gcn tra:combined_pe@@@transforms.CombinedPSEs.encodings=[LapPE,RWSE,ElectrostaticPE,HKdiagSE] num:1 hid:128)
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/CombinedPSEs/2691934090
[8/18] SUCCESS | Total: 81,060 (mod:gcn tra:combined_pe@@@transforms.CombinedPSEs.encodings=[LapPE,RWSE,ElectrostaticPE,HKdiagSE] num:1 hid:256)
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/CombinedPSEs/2691934090
[9/18] SUCCESS | Total: 40,740 (mod:gcn tra:combined_pe@@@transforms.CombinedPSEs.encodings=[LapPE,RWSE,ElectrostaticPE,HKdiagSE] num:2 hid:128)
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/CombinedPSEs/2691934090
[10/18] SUCCESS | Total: 146,852 (mod:gcn tra:combined_pe@@@transforms.CombinedPSEs.encodings=[LapPE,RWSE,Electr

Processing...



Applying transforms to 188 graphs...


Processing graphs: 100%|██████████| 188/188 [00:31<00:00,  6.05graph/s]
Done!


[13/18] SUCCESS | Total: 22,263 (mod:gcn tra:combined_fe@@@transforms.CombinedFEs.encodings=[HKFE,KHopFE,PPRFE] num:1 hid:128)
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/CombinedFEs/4237887575
[14/18] SUCCESS | Total: 77,175 (mod:gcn tra:combined_fe@@@transforms.CombinedFEs.encodings=[HKFE,KHopFE,PPRFE] num:1 hid:256)
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/CombinedFEs/4237887575
[15/18] SUCCESS | Total: 38,775 (mod:gcn tra:combined_fe@@@transforms.CombinedFEs.encodings=[HKFE,KHopFE,PPRFE] num:2 hid:128)
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/CombinedFEs/4237887575
[16/18] SUCCESS | Total: 142,967 (mod:gcn tra:combined_fe@@@transforms.CombinedFEs.encodings=[HKFE,KHopFE,PPRFE] num:2 hid:256)
Transform parameters are the same, using 